# CLIP & LLaVA — TP complet et corrigé pour Google Colab


Ce notebook contient l'énoncé fourni, puis les implémentations des 13 exercices,
les réponses aux 36 questions et le mini-projet.

## Démarrage

1. Importer ce fichier dans [Google Colab](https://colab.research.google.com/).
2. Choisir **Exécution → Modifier le type d'exécution → GPU** pour LLaVA.

4. Exécuter d'abord l'installation. Si Colab demande un redémarrage, redémarrer
   puis reprendre à la configuration, sans réinstaller en boucle.
5. CLIP fonctionne aussi sur CPU. Le parcours LLaVA utilise ici un GPU CUDA
   et une quantification 4 bits ; il est ignoré explicitement si CUDA est absent.
6. Remplir les évaluations humaines VQA et de recherche, puis relancer leur calcul
   et l'export final. Télécharger aussi le notebook **avec ses sorties**.

Les poids LLaVA représentent plusieurs Go à télécharger, même en chargement quantifié.
La compatibilité et la mémoire dépendent de l'environnement ; une T4 est une cible
pratique pour ce parcours 4 bits, sans garantie pour tout état de session.

**Données par défaut :** un petit échantillon équilibré du jeu de test CIFAR-10
(100 images, 10 classes), plus cinq scènes COCO pour LLaVA. Des images personnelles
peuvent remplacer ces données. CIFAR-10 (32 × 32) convient à une démonstration,
pas à une estimation générale des performances ou à une VQA détaillée.

**Vérification de cette livraison :** structure et syntaxe contrôlées ; les modèles
ne sont pas exécutés ici. Les expériences GPU et leurs résultats sont à produire dans Colab.


# LAB — Vision-Language Models: CLIP & LLaVA

**Module:** Artificial Intelligence / Generative AI  
**Level:** Phd Students  
**Environment:** Google Colab / Kaggle / Jupyter Notebook  
**Frameworks:** Python, PyTorch, Hugging Face Transformers

## Videos (Must see before Starting the Lab)
* https://www.youtube.com/watch?v=jXD6O93Ptks&t=6s
* https://www.youtube.com/watch?v=bK9ns4DkxQg&t=8s

## Objectives

This practical work introduces two important vision-language models:

- **CLIP**: image-text representation, similarity, zero-shot classification and image retrieval.
- **LLaVA**: multimodal image understanding and visual question answering.

At the end of the Lab, students should be able to:
1. Load a pretrained CLIP model.
2. Compare images and textual descriptions.
3. Perform zero-shot classification.
4. Evaluate classification accuracy.
5. Perform text-to-image retrieval.
6. Understand the architecture of LLaVA.
7. Run visual question answering.
8. Identify visual hallucinations.
9. Compare CLIP and LLaVA.
10. Build a small multimodal assistant.

> **Notebook organization:** Part I is the student statement. Part II contains the correction and reference implementations. Students should first complete the exercises without opening the correction section.


# Part II — Corrigé exécutable

## 0. Installation

Cette version vise `transformers==4.57.1` pour stabiliser les API du TP.
Elle conserve PyTorch et Torchvision fournis ensemble par Colab. Sur un Jupyter
local, installer d'abord une paire compatible depuis le site officiel de PyTorch.
PyTorch ≥ 2.6 est requis ici pour le chargement du checkpoint CLIP au format PyTorch.
L'installation ne télécharge pas encore les poids des modèles.


In [2]:
import subprocess
import sys

packages = [
    "transformers==4.57.1", "accelerate>=1.0,<2", "bitsandbytes>=0.45,<1",
    "sentencepiece>=0.2,<1", "pillow", "matplotlib", "pandas", "requests",
    "ipywidgets>=8,<9", "librosa>=0.10.2", "packaging"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Installation terminée. Redémarrer uniquement si l'environnement le demande.")


Installation terminée. Redémarrer uniquement si l'environnement le demande.


## 1. Configuration et traçabilité

Modifier `ROOT` pour un dossier persistant, par exemple dans Drive **après** l'avoir
monté manuellement. Aucun montage Drive n'est nécessaire pour exécuter le TP.
Sur Kaggle, activer Internet et le GPU dans les paramètres du notebook.
`RUN_LLAVA=False` permet de travailler seulement sur CLIP.


In [3]:
import gc
import hashlib
import importlib.metadata as metadata
import json
import platform
import random
import shutil
import time
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image, ImageOps
from IPython.display import display, Markdown, clear_output
from packaging.version import Version
from transformers import CLIPModel, CLIPProcessor

ROOT = Path("/kaggle/working/clip_llava_lab" if Path("/kaggle").exists() else "clip_llava_lab").resolve()
DATA_DIR = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
for folder in [DATA_DIR, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 2026
N_PER_CLASS = 10  # 100 images : minimum 1 pour satisfaire les 10 images de l'exercice 5
BATCH_SIZE = 8
RUN_LLAVA = True
USE_4BIT = True
MAX_NEW_TOKENS = 128
CLIP_ID = "openai/clip-vit-base-patch32"
LLAVA_ID = "llava-hf/llava-1.5-7b-hf"
CLIP_REVISION = "main"  # Remplacer par le commit enregistré pour une réexécution figée.
LLAVA_REVISION = "main"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if Version(torch.__version__.split("+")[0]) < Version("2.6"):
    raise RuntimeError("Utiliser un runtime Colab récent : PyTorch >= 2.6 requis pour ce parcours.")

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

manifest = {
    "seed": SEED, "python": platform.python_version(), "torch": torch.__version__,
    "packages": {name: package_version(name) for name in [
        "transformers", "torchvision", "accelerate", "bitsandbytes", "numpy", "pandas"]},
    "device": str(device), "clip_id": CLIP_ID, "llava_id": LLAVA_ID,
    "clip_revision_requested": CLIP_REVISION, "llava_revision_requested": LLAVA_REVISION,
    "run_llava": RUN_LLAVA, "use_4bit": USE_4BIT,
    "n_per_class": N_PER_CLASS, "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS, "llava_status": "not_started"
}
if torch.cuda.is_available():
    manifest["gpu"] = torch.cuda.get_device_name(0)
    manifest["vram_total_gib"] = torch.cuda.get_device_properties(0).total_memory / 2**30

def save_manifest():
    (RESULTS / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

def open_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert("RGB")

def image_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

save_manifest()
print("Device :", device, "| Résultats :", RESULTS)
print("Versions :", manifest["packages"])


Device : cuda:0 | Résultats : /kaggle/working/clip_llava_lab/results
Versions : {'transformers': '4.57.1', 'torchvision': '0.26.0+cu130', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'numpy': '2.1.3', 'pandas': '2.2.3'}


## 2. Images étiquetées pour les exercices CLIP

**Mode automatique :** échantillonnage aléatoire équilibré du test CIFAR-10 avec
indices sauvegardés. Aucun entraînement n'est réalisé. Le téléchargement initial
du paquet CIFAR-10 complet est plus volumineux que l'échantillon retenu.

**Mode personnel :** créer `data/custom_labeled/<classe>/*.jpg`, avec au moins
10 images au total et des noms de classes en anglais (`cat`, `dog`, etc.), puis
mettre `USE_CUSTOM_DATA=True`. Les noms de dossiers constituent les annotations
humaines et ne doivent pas provenir des prédictions CLIP.


In [ ]:
USE_CUSTOM_DATA = False
CUSTOM_DIR = DATA_DIR / "custom_labeled"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
records = []

if USE_CUSTOM_DATA:
    for class_dir in sorted(CUSTOM_DIR.iterdir()):
        if class_dir.is_dir():
            for path in sorted(class_dir.rglob("*")):
                if path.suffix.lower() in IMAGE_EXTENSIONS:
                    open_rgb(path)  # Validation des fichiers avant toute évaluation.
                    records.append({"path": str(path), "actual_class": class_dir.name,
                                    "dataset": "custom", "source_index": None})
    classes = sorted({r["actual_class"] for r in records})
else:
    try:
        from torchvision.datasets import CIFAR10
    except (ImportError, RuntimeError) as exc:
        raise RuntimeError("Torch/Torchvision incompatibles : utiliser un runtime Colab neuf, "
                           "ou installer une paire compatible depuis pytorch.org.") from exc
    cifar = CIFAR10(root=str(DATA_DIR), train=False, download=True)
    classes = list(cifar.classes)
    rng = np.random.default_rng(SEED)
    labels = np.asarray(cifar.targets)
    if not 1 <= N_PER_CLASS <= 1000:
        raise ValueError("N_PER_CLASS doit être entre 1 et 1000.")
    for label, name in enumerate(classes):
        for index in rng.choice(np.flatnonzero(labels == label), N_PER_CLASS, replace=False):
            path = DATA_DIR / "cifar_sample" / name / f"test_{int(index):05d}.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            cifar[int(index)][0].save(path)
            records.append({"path": str(path), "actual_class": name,
                            "dataset": "CIFAR10-test", "source_index": int(index)})
    del cifar

if len(records) < 10 or len(classes) < 2:
    raise ValueError("Il faut au moins 10 images étiquetées et deux classes candidates.")
dataset = [(r["path"], r["actual_class"]) for r in records]
dataset_df = pd.DataFrame(records)
dataset_df["sha256"] = [image_hash(r["path"]) for r in records]
dataset_df.to_csv(RESULTS / "dataset_manifest.csv", index=False)
print(len(dataset), "images | Classes :", classes)
display(dataset_df.groupby("actual_class").size().rename("n").to_frame())

# Facultatif : chemin vers votre propre image pour les exercices 2, 3, 4 et 6.
IMAGE_PATH = None
default_path = next((r["path"] for r in records if r["actual_class"] == "cat"), records[0]["path"])
IMAGE_PATH = str(IMAGE_PATH or default_path)
image = open_rgb(IMAGE_PATH)
print("Q5 — taille (largeur, hauteur) :", image.size, "| Q6 — mode :", image.mode)
display(image.resize((320, 320)))


 33%|███▎      | 56.7M/170M [19:39<29:31, 64.2kB/s]

## 3. Exercices 1 et 3 — Charger CLIP et comparer image/texte

Le softmax produit une distribution **relative aux textes candidats**. Il ne donne
pas une probabilité calibrée que l'image soit correctement reconnue.


In [ ]:
model = CLIPModel.from_pretrained(CLIP_ID, revision=CLIP_REVISION).to(device).eval()
processor = CLIPProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
manifest["clip_commit"] = getattr(model.config, "_commit_hash", None)
save_manifest()

def clip_scores(im, texts):
    if not texts or any(not str(t).strip() for t in texts):
        raise ValueError("La liste de textes doit être non vide et sans texte vide.")
    inputs = processor(text=list(texts), images=im.convert("RGB"),
                       return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits_per_image[0].float()
    return logits.cpu(), logits.softmax(dim=-1).cpu()

texts = ["a photo of a cat", "a photo of a dog", "a photo of a car",
         "a photo of a horse", "a photo of a person"]
logits, probs = clip_scores(image, texts)
similarity_df = pd.DataFrame({"text": texts, "logit": logits.numpy(), "relative_probability": probs.numpy()})
similarity_df.to_csv(RESULTS / "image_text_similarity.csv", index=False)
display(similarity_df.sort_values("relative_probability", ascending=False))
print("Q7 :", texts[int(probs.argmax())])
print("Q8 : comparer visuellement avec l'image affichée ; le modèle ne valide pas sa propre réponse.")


## 4. Exercices 4 et 5 — Classification zero-shot et accuracy

Les textes de classes restent identiques pour toutes les images. `automobile`
est le nom CIFAR-10 de la voiture. La fonction renvoie la classe et son score
softmax relatif. Aucun rejet « inconnu » n'est implémenté.


In [ ]:
def class_prompts(candidate_classes):
    if not candidate_classes or len(set(candidate_classes)) != len(candidate_classes):
        raise ValueError("Classes candidates non vides et uniques requises.")
    if any(not str(c).strip() for c in candidate_classes):
        raise ValueError("Nom de classe vide.")
    return [f"a photo of a {c}" for c in candidate_classes]

def classify_image(image, classes):
    _, probabilities = clip_scores(image, class_prompts(classes))
    index = int(probabilities.argmax())
    return classes[index], float(probabilities[index])

prediction, probability = classify_image(image, classes)
print("Exercice 4 — Prediction :", prediction, "| score relatif :", round(probability, 4))

prompts_classes = class_prompts(classes)
rows = []
start = time.perf_counter()
for begin in range(0, len(dataset), BATCH_SIZE):
    batch = dataset[begin:begin + BATCH_SIZE]
    batch_images = [open_rgb(path) for path, _ in batch]
    inputs = processor(text=prompts_classes, images=batch_images,
                       return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        probabilities = model(**inputs).logits_per_image.float().softmax(dim=-1).cpu()
    for (path, actual), prob in zip(batch, probabilities):
        index = int(prob.argmax())
        predicted = classes[index]
        rows.append({"image": path, "actual_class": actual, "predicted_class": predicted,
                     "relative_probability": float(prob[index]), "correct": predicted == actual})
eval_df = pd.DataFrame(rows)
clip_accuracy = float(eval_df["correct"].mean())
manifest["classification_seconds"] = time.perf_counter() - start
manifest["clip_accuracy"] = clip_accuracy
manifest["candidate_classes"] = classes
save_manifest()
eval_df.to_csv(RESULTS / "clip_classification.csv", index=False)
print(f"Q14 — Accuracy : {eval_df.correct.sum()}/{len(eval_df)} = {clip_accuracy:.2%}")
display(eval_df)
print("Q15 — Erreurs :")
display(eval_df.loc[~eval_df["correct"]])

confusion = pd.crosstab(eval_df.actual_class, eval_df.predicted_class).reindex(
    index=classes, columns=classes, fill_value=0)
confusion.to_csv(RESULTS / "clip_confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(9, 7))
plot = ax.imshow(confusion.to_numpy(), cmap="Blues")
ax.set(xticks=range(len(classes)), yticks=range(len(classes)),
       xticklabels=classes, yticklabels=classes, xlabel="Prédiction", ylabel="Vérité terrain",
       title=f"CLIP zero-shot — n={len(eval_df)}, accuracy={clip_accuracy:.1%}")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.colorbar(plot, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES / "clip_confusion_matrix.png", dpi=160)
plt.show()


## 5. Exercice 6 — Influence des prompts

La comparaison porte sur la même image. Les cinq prompts décrivent tous un chat :
leurs scores softmax se partagent donc une masse de 1, ils ne représentent pas
cinq classes distinctes. Ne pas sélectionner un prompt sur les images test puis
présenter le score sur ces mêmes images comme une évaluation indépendante.


In [ ]:
prompts = ["a cat", "a photo of a cat", "a photograph showing a cat",
           "an image containing a cat", "a domestic animal called a cat"]
logits, probabilities = clip_scores(image, prompts)
prompt_df = pd.DataFrame({"prompt": prompts, "logit": logits.numpy(),
                          "relative_probability": probabilities.numpy()})
prompt_df.to_csv(RESULTS / "prompt_comparison.csv", index=False)
display(prompt_df.sort_values("logit", ascending=False))
print("Q17 — Logits tous égaux ?", bool(np.allclose(logits.numpy(), logits.numpy()[0])))
print("Q18 — Meilleur prompt pour cette image :", prompts[int(logits.argmax())])


## 6. Exercices 7 et 8 — Base d'images et recherche Top-5

Le dossier `data/retrieval_custom` peut contenir vos images supplémentaires, notamment
des scènes de football. La base par défaut utilise l'échantillon CIFAR-10 ; aucun label
n'est transmis à l'encodeur. Elle doit contenir au moins 20 images.

La requête de l'énoncé est conservée. **CIFAR-10 ne contient pas de catégorie football
ni personne :** un Top-5 sera toujours renvoyé même si aucune image n'est pertinente.
Ajouter des scènes adaptées pour évaluer cette requête. Une seconde recherche `a cat`
illustre le fonctionnement sur une catégorie présente. `relevant` est à renseigner
manuellement (0/1), sans supposer que le rang 1 est correct.


In [ ]:
RETRIEVAL_CUSTOM = DATA_DIR / "retrieval_custom"
RETRIEVAL_CUSTOM.mkdir(exist_ok=True)
extra_paths = sorted(p for p in RETRIEVAL_CUSTOM.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
retrieval_paths = list(dict.fromkeys([r["path"] for r in records] + [str(p) for p in extra_paths]))
if len(retrieval_paths) < 20:
    raise ValueError("Exercice 7 : fournir au moins 20 images pour la recherche.")

def encode_images(paths, batch_size=BATCH_SIZE):
    vectors = []
    for begin in range(0, len(paths), batch_size):
        batch_images = [open_rgb(p) for p in paths[begin:begin + batch_size]]
        inputs = processor(images=batch_images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            feats = model.get_image_features(**inputs).float()
            feats = torch.nn.functional.normalize(feats, dim=-1)
        vectors.append(feats.cpu())
    return torch.cat(vectors, dim=0)

def encode_text(query):
    if not str(query).strip():
        raise ValueError("Requête vide.")
    inputs = processor(text=[query], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        feats = model.get_text_features(**inputs).float()
        feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats.cpu()

image_embeddings = encode_images(retrieval_paths)
np.save(RESULTS / "clip_image_embeddings.npy", image_embeddings.numpy())
retrieval_manifest = pd.DataFrame({"embedding_row": range(len(retrieval_paths)), "image": retrieval_paths,
                                   "sha256": [image_hash(p) for p in retrieval_paths]})
retrieval_manifest.to_csv(RESULTS / "retrieval_database.csv", index=False)

def retrieve_images(query, k=5):
    if k < 1:
        raise ValueError("k doit être >= 1.")
    similarities = (image_embeddings @ encode_text(query).T).squeeze(1)
    values, indices = similarities.topk(min(k, len(retrieval_paths)))
    return pd.DataFrame({"rank": range(1, len(indices) + 1), "query": query,
                         "image": [retrieval_paths[int(i)] for i in indices],
                         "cosine_similarity": values.numpy()})

def show_retrieval(table, filename=None):
    fig, axes = plt.subplots(1, len(table), figsize=(3 * len(table), 3.4), squeeze=False)
    for ax, row in zip(axes[0], table.itertuples()):
        ax.imshow(open_rgb(row.image))
        ax.set_title(f"#{row.rank} | cos={row.cosine_similarity:.3f}")
        ax.axis("off")
    fig.suptitle(table.iloc[0]["query"])
    fig.tight_layout()
    if filename:
        fig.savefig(FIGURES / filename, dpi=160)
    plt.show()

QUERY = "a person playing football"
top5 = retrieve_images(QUERY)
top5["relevant"] = pd.NA
top5.to_csv(RESULTS / "retrieval_football_top5.csv", index=False)
show_retrieval(top5, "retrieval_football_top5.png")
display(top5)
print("Q20 — Premier résultat :", top5.iloc[0]["image"])
print("Q21 — À juger visuellement ; score élevé ne signifie pas pertinence garantie.")
top5_cat = retrieve_images("a photo of a cat")
top5_cat.to_csv(RESULTS / "retrieval_cat_top5.csv", index=False)
show_retrieval(top5_cat, "retrieval_cat_top5.png")


### Annotation facultative de la recherche

Après inspection, éditer le CSV dans Colab, enregistrer 0 ou 1 dans `relevant`,
puis relancer cette cellule. On distingue la proportion annotée et la Precision@5
sur les cinq résultats ; aucune métrique complète n'est calculée sur des cases vides.


In [ ]:
relevance_df = pd.read_csv(RESULTS / "retrieval_football_top5.csv")
rel = pd.to_numeric(relevance_df["relevant"], errors="raise")
if not rel.dropna().isin([0, 1]).all():
    raise ValueError("relevant doit contenir 0, 1 ou une case vide.")
print(f"Recherche : {rel.notna().sum()}/{len(rel)} résultats annotés.")
if rel.notna().all():
    manifest["retrieval_precision_at_5"] = float(rel.mean())
    print("Precision@5 :", float(rel.mean()))
else:
    manifest["retrieval_precision_at_5"] = None
    print("Precision@5 en attente de l'évaluation humaine.")
save_manifest()


## 7. Exercice 9 — Architecture LLaVA 1.5

**Image → encodeur visuel CLIP ViT-L/14 → projecteur multimodal MLP →
modèle de langage Vicuna → réponse textuelle.**

L'encodeur visuel extrait des représentations, le projecteur les aligne sur
l'espace des embeddings du modèle de langage, et le modèle de langage génère
la réponse à partir de l'image et de la question. Le CLIP ViT-B/32 chargé plus haut
est un modèle distinct de l'encodeur visuel intégré dans LLaVA 1.5.

## 8. Images de scènes pour LLaVA

Placer cinq images lisibles dans `data/vqa_custom` pour employer vos propres scènes.
À défaut, cinq images COCO val2017 sont téléchargées depuis la source officielle.
Choisir des scènes variées, noter les sources et observer les images **avant** de
noter les réponses. Les petites images CIFAR-10 ne servent pas de vérité terrain VQA.


In [ ]:
VQA_CUSTOM = DATA_DIR / "vqa_custom"
VQA_CUSTOM.mkdir(exist_ok=True)
vqa_paths = sorted(p for p in VQA_CUSTOM.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
scene_sources = []
if RUN_LLAVA and len(vqa_paths) < 5:
    COCO_IDS = ["000000039769", "000000397133", "000000037777", "000000252219", "000000087038"]
    scene_dir = DATA_DIR / "coco_scenes"
    scene_dir.mkdir(exist_ok=True)
    for coco_id in COCO_IDS:
        url = f"https://images.cocodataset.org/val2017/{coco_id}.jpg"
        path = scene_dir / f"{coco_id}.jpg"
        try:
            if not path.exists():
                response = requests.get(url, timeout=60)
                response.raise_for_status()
                Image.open(BytesIO(response.content)).convert("RGB").save(path)
            open_rgb(path)
            scene_sources.append({"image": str(path), "source_url": url})
        except (requests.RequestException, OSError) as exc:
            print("Scène indisponible :", coco_id, type(exc).__name__)
    vqa_paths = list(vqa_paths) + [Path(r["image"]) for r in scene_sources]
vqa_paths = list(dict.fromkeys(vqa_paths))[:5]
if vqa_paths:
    sources_by_path = {r["image"]: r["source_url"] for r in scene_sources}
    pd.DataFrame([{"image": str(p), "source": sources_by_path.get(str(p), "custom"),
                   "sha256": image_hash(p)} for p in vqa_paths]).to_csv(RESULTS / "vqa_images.csv", index=False)
    fig, axes = plt.subplots(1, len(vqa_paths), figsize=(4 * len(vqa_paths), 4), squeeze=False)
    for ax, path in zip(axes[0], vqa_paths):
        ax.imshow(open_rgb(path)); ax.set_title(path.name); ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURES / "vqa_images.png", dpi=140)
    plt.show()
if len(vqa_paths) < 5:
    print("Exercice 11 : fournir cinq images dans", VQA_CUSTOM, "puis relancer cette cellule.")


## 9. Exercice 10 — Charger et interroger LLaVA

CLIP est déplacé sur CPU pour laisser la mémoire GPU à LLaVA ; ses embeddings de
recherche restent réutilisables. La quantification 4 bits réduit la mémoire des poids,
elle peut modifier les réponses par rapport au modèle en demi-précision. La
configuration et les commits réellement chargés sont enregistrés dans le manifeste.

Les erreurs de chargement sont affichées et enregistrées. Les expériences LLaVA
restent alors signalées comme non exécutées, sans réponses de substitution.


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration

model.to("cpu")
device = torch.device("cpu")  # Les fonctions CLIP continuent à fonctionner pour le bonus.
if "inputs" in globals():
    del inputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

llava = None
processor_llava = None
if RUN_LLAVA and torch.cuda.is_available():
    try:
        compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        kwargs = {"revision": LLAVA_REVISION, "dtype": compute_dtype,
                  "device_map": {"": 0}, "low_cpu_mem_usage": True}
        if USE_4BIT:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
        processor_llava = AutoProcessor.from_pretrained(LLAVA_ID, revision=LLAVA_REVISION)
        llava = LlavaForConditionalGeneration.from_pretrained(LLAVA_ID, **kwargs).eval()
        processor_llava.tokenizer.padding_side = "left"
        processor_llava.patch_size = llava.config.vision_config.patch_size
        processor_llava.vision_feature_select_strategy = llava.config.vision_feature_select_strategy
        processor_llava.num_additional_image_tokens = 1  # CLIP ajoute un jeton CLS.
        manifest["llava_commit"] = getattr(llava.config, "_commit_hash", None)
        manifest["llava_dtype"] = str(compute_dtype)
        manifest["llava_status"] = "loaded"
        print("LLaVA chargé :", LLAVA_ID, "| 4 bits :", USE_4BIT)
    except Exception as exc:
        llava = None
        gc.collect()
        torch.cuda.empty_cache()
        manifest["llava_status"] = "load_failed"
        manifest["llava_error"] = f"{type(exc).__name__}: {exc}"
        print("Chargement LLaVA échoué :", manifest["llava_error"])
        print("Libérer la mémoire, vérifier les téléchargements, puis relancer cette cellule.")
else:
    manifest["llava_status"] = "disabled" if not RUN_LLAVA else "skipped_no_cuda"
    print("LLaVA non exécuté :", manifest["llava_status"])
save_manifest()

def ask_llava(image_or_path, question, max_new_tokens=MAX_NEW_TOKENS):
    if llava is None:
        raise RuntimeError("LLaVA n'est pas chargé. Activer CUDA et relancer le chargement.")
    if not str(question).strip():
        raise ValueError("Question vide.")
    im = image_or_path.convert("RGB") if isinstance(image_or_path, Image.Image) else open_rgb(image_or_path)
    conversation = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor_llava.apply_chat_template(conversation, add_generation_prompt=True)
    inputs_llava = processor_llava(images=im, text=prompt, return_tensors="pt")
    target = llava.get_input_embeddings().weight.device
    inputs_llava = {k: (v.to(device=target, dtype=compute_dtype) if v.is_floating_point()
                        else v.to(device=target)) for k, v in inputs_llava.items()}
    prompt_length = inputs_llava["input_ids"].shape[1]
    with torch.inference_mode():
        generated = llava.generate(**inputs_llava, max_new_tokens=max_new_tokens, do_sample=False)
    return processor_llava.batch_decode(generated[:, prompt_length:], skip_special_tokens=True)[0].strip()

demo_questions = ["What is happening in this image?", "How many people are visible?",
                  "What objects can you identify?", "Describe the image in detail."]
if llava is not None and vqa_paths:
    demo_rows = []
    for question in demo_questions:
        answer = ask_llava(vqa_paths[0], question)
        demo_rows.append({"image": str(vqa_paths[0]), "question": question, "llava_answer": answer})
        print("Q:", question, "\nA:", answer, "\n")
        pd.DataFrame(demo_rows).to_csv(RESULTS / "llava_demo.csv", index=False)
else:
    print("Exercice 10 en attente de LLaVA et d'une image.")


## 10. Exercice 11 — VQA sur 5 images × 5 questions

Les 25 réponses sont sauvegardées au fur et à mesure. En cas d'arrêt, relancer la
cellule reprend les questions absentes sans effacer les annotations déjà saisies.
Pour recommencer une expérience après changement de modèle ou de paramètres,
utiliser un nouveau `ROOT` ou archiver puis supprimer ce CSV ; ne pas mélanger les runs.

**Barème binaire défini avant de lire les réponses :** 1 si la réponse est correcte
sur tous les éléments explicitement demandés et n'ajoute pas d'affirmation visuelle
fausse ; 0 sinon. Pour le comptage, demander le nombre exact de personnes visibles.
Pour les objets, vérifier les objets effectivement cités. Si l'image ne permet pas
de conclure, une réponse prudente peut être correcte ; documenter la règle dans `notes`.
Noter d'abord une référence humaine dans `reference_answer`, puis 0/1 dans
`human_correct`. `human_correct` vide signifie **non évalué**, pas incorrect.

Ce score est une accuracy pédagogique évaluée humainement ; il ne correspond pas
au score officiel VQA avec plusieurs annotateurs. Cinq images ne constituent pas
un benchmark représentatif. Un second évaluateur peut réduire la subjectivité.


In [ ]:
VQA_FILE = RESULTS / "vqa_evaluation.csv"
vqa_questions = ["How many people are visible?", "What objects can you see?",
                 "Where are the people? If there are none, say so.",
                 "What are they doing? If there are no people, say so.",
                 "What is visible on the board? If no board is visible, say so."]
vqa_rows = pd.read_csv(VQA_FILE).to_dict("records") if VQA_FILE.exists() else []
existing_keys = {(str(r["image"]), r["question"]) for r in vqa_rows}
if llava is not None and len(vqa_paths) == 5:
    for path in vqa_paths:
        for question in vqa_questions:
            if (str(path), question) in existing_keys:
                continue
            answer = ask_llava(path, question)
            vqa_rows.append({"image": str(path), "question": question, "llava_answer": answer,
                             "reference_answer": "", "human_correct": None, "notes": ""})
            pd.DataFrame(vqa_rows).to_csv(VQA_FILE, index=False)
            print(f"{len(vqa_rows)}/25 — {path.name} — {answer[:100]}")
    display(pd.DataFrame(vqa_rows))
else:
    print("VQA non exécutée : il faut LLaVA chargé et exactement cinq images sélectionnées.")


### Interface de notation humaine

Choisir une ligne, observer l'image, saisir la référence puis le score et enregistrer.
Chaque sauvegarde met à jour le CSV. Laisser « non évalué » si l'observation est
encore incertaine. Relancer le calcul qui suit après les 25 notations.


In [ ]:
import ipywidgets as widgets

if VQA_FILE.exists():
    grading_df = pd.read_csv(VQA_FILE)
    for column in ["reference_answer", "notes"]:
        grading_df[column] = grading_df[column].astype(object)
    selector = widgets.Dropdown(options=[(f"{i+1}: {Path(r.image).name} — {r.question}", i)
                                         for i, r in grading_df.iterrows()], description="Question")
    reference_box = widgets.Textarea(description="Référence", layout=widgets.Layout(width="95%"))
    grade_box = widgets.Dropdown(options=[("Non évalué", ""), ("Correct : 1", "1"), ("Incorrect : 0", "0")],
                                 description="Score")
    notes_box = widgets.Textarea(description="Notes", layout=widgets.Layout(width="95%"))
    save_button = widgets.Button(description="Enregistrer la note", button_style="success")
    grading_view = widgets.Output()
    grading_status = widgets.Output()

    def load_grade(change=None):
        row = grading_df.loc[selector.value]
        reference_box.value = "" if pd.isna(row.reference_answer) else str(row.reference_answer)
        notes_box.value = "" if pd.isna(row.notes) else str(row.notes)
        grade_box.value = "" if pd.isna(row.human_correct) else str(int(row.human_correct))
        with grading_view:
            clear_output(wait=True)
            im = open_rgb(row.image); im.thumbnail((650, 420)); display(im)
            print("Question :", row.question, "\nRéponse LLaVA :", row.llava_answer)

    def save_grade(_):
        i = selector.value
        grading_df.loc[i, "reference_answer"] = reference_box.value
        grading_df.loc[i, "notes"] = notes_box.value
        grading_df.loc[i, "human_correct"] = np.nan if grade_box.value == "" else int(grade_box.value)
        grading_df.to_csv(VQA_FILE, index=False)
        with grading_status:
            clear_output(wait=True)
            print("Note sauvegardée pour la ligne", i + 1)

    selector.observe(load_grade, names="value")
    save_button.on_click(save_grade)
    display(widgets.VBox([selector, grading_view, reference_box, grade_box, notes_box,
                          save_button, grading_status]))
    load_grade()
else:
    print("La grille apparaîtra après l'exécution des 25 questions.")


In [ ]:
vqa_accuracy = None
if VQA_FILE.exists():
    vqa_df = pd.read_csv(VQA_FILE)
    grades = pd.to_numeric(vqa_df["human_correct"], errors="raise")
    if not grades.dropna().isin([0, 1]).all():
        raise ValueError("human_correct doit être 0, 1 ou vide.")
    n_done = int(grades.notna().sum())
    print(f"Questions annotées : {n_done}/{len(vqa_df)}")
    if n_done:
        print(f"Accuracy sur les réponses annotées seulement : {grades.dropna().mean():.2%}")
    if len(vqa_df) == 25 and n_done == 25:
        vqa_accuracy = float(grades.mean())
        print(f"Q27 — Accuracy VQA complète : {int(grades.sum())}/25 = {vqa_accuracy:.2%}")
    else:
        print("Q27 — Accuracy complète en attente des 25 réponses et des 25 notes humaines.")
else:
    print("Q27 — Pas de résultats VQA : expérience non exécutée.")
manifest["vqa_accuracy_25"] = vqa_accuracy
save_manifest()


## 11. Exercice 12 — Test d'hallucination

Choisir une image sans vélo et confirmer son absence après inspection.
Par défaut, la première scène est proposée, mais **l'absence n'est pas validée
automatiquement**. Les trois questions sont posées dans des conversations
indépendantes pour isoler leur formulation. Une version avec historique serait
une autre condition expérimentale.

Une question présupposant un vélo peut induire une hallucination, mais elle ne
la garantit pas. Si LLaVA refuse correctement la prémisse, le rapport indique
« aucune hallucination observée dans ce test ». Pour le livrable demandant au
moins un exemple, essayer d'autres images/questions et rapporter un cas réel
**s'il est observé**, en indiquant le nombre de tentatives. Ne pas fabriquer d'exemple.


In [ ]:
HALLUCINATION_IMAGE = str(vqa_paths[0]) if vqa_paths else None
HALLUCINATION_ABSENCE_CONFIRMED = False  # Mettre True uniquement après inspection humaine.
hallucination_questions = ["Is there a bicycle in this image?", "What color is the bicycle?",
                           "Where is the bicycle located?"]
HALLUCINATION_FILE = RESULTS / "hallucination_test.csv"
if HALLUCINATION_IMAGE:
    display(open_rgb(HALLUCINATION_IMAGE))
if llava is not None and HALLUCINATION_IMAGE:
    if not HALLUCINATION_ABSENCE_CONFIRMED:
        print("Inspecter l'image, confirmer l'absence du vélo, puis relancer cette cellule.")
    else:
        old_hall = pd.read_csv(HALLUCINATION_FILE).to_dict("records") if HALLUCINATION_FILE.exists() else []
        attempt_id = 1 + max([int(r.get("attempt_id", 0)) for r in old_hall], default=0)
        for question in hallucination_questions:
            answer = ask_llava(HALLUCINATION_IMAGE, question)
            old_hall.append({"attempt_id": attempt_id, "image": HALLUCINATION_IMAGE,
                             "bicycle_absent_human_confirmed": True,
                             "question": question, "llava_answer": answer,
                             "human_hallucination": None, "notes": ""})
            pd.DataFrame(old_hall).to_csv(HALLUCINATION_FILE, index=False)
            print("Q:", question, "\nA:", answer, "\n")
        display(pd.DataFrame(old_hall))
        print("Noter human_hallucination : 1 si une information visuelle fausse est inventée, 0 sinon.")
else:
    print("Test d'hallucination non exécuté.")


## 12. Exercice 13 — Comparaison sur la même image

Utiliser de préférence une image d'un chat, chien ou cheval. La liste CLIP est
fermée : si l'image montre un autre animal, il sera néanmoins forcé dans une de
ces classes. Les deux modèles voient ici exactement la même image source.


In [ ]:
comparison_classes = ["cat", "dog", "horse"]
comparison_prediction, comparison_probability = classify_image(image, comparison_classes)
comparison_answer = ask_llava(image, "What animal is present in the image? Describe its appearance.") if llava is not None else None
comparison_result = {"image": IMAGE_PATH, "clip_prediction": comparison_prediction,
                      "clip_relative_probability": comparison_probability,
                      "llava_answer": comparison_answer}
(RESULTS / "same_image_comparison.json").write_text(json.dumps(comparison_result, indent=2, ensure_ascii=False))
print(comparison_result)

comparison_table = pd.DataFrame([
    ["Input", "Images + textes candidats", "Image + question/instruction"],
    ["Main output", "Embeddings et scores de similarité", "Texte généré"],
    ["Zero-shot classification", "Directe, liste de classes fournie", "Possible par instruction, format moins contraint"],
    ["Image retrieval", "Adapté, embeddings indexables", "Pas son usage direct principal"],
    ["Text generation", "Non", "Oui"],
    ["VQA", "Choix de réponses par similarité, limité", "Réponses libres à des questions visuelles"],
    ["Open-ended response", "Non", "Oui"]
], columns=["Property", "CLIP", "LLaVA"])
comparison_table.to_csv(RESULTS / "clip_llava_comparison.csv", index=False)
display(comparison_table)


## 13. Mini-projet — Assistant image + question

Sélectionner une image de la liste ou importer une image locale. Le bouton
interroge LLaVA et enregistre la réponse ; l'application reste dans le notebook.
Les réponses sont des observations du modèle, pas des annotations fiables par défaut.


In [ ]:
assistant_log = []
choices = list(dict.fromkeys([IMAGE_PATH] + [str(p) for p in vqa_paths]))
image_choice = widgets.Dropdown(options=[(Path(p).name, p) for p in choices], description="Image")
image_upload = widgets.FileUpload(accept="image/*", multiple=False, description="Importer")
question_box = widgets.Textarea(value="Describe this image and mention what you cannot determine.",
                                description="Question", layout=widgets.Layout(width="95%"))
answer_button = widgets.Button(description="Interroger LLaVA", button_style="primary", disabled=llava is None)
assistant_output = widgets.Output()

def run_assistant(_):
    answer_button.disabled = True
    with assistant_output:
        clear_output(wait=True)
        try:
            if image_upload.value:
                item = image_upload.value[0]
                uploaded_image = Image.open(BytesIO(bytes(item["content"]))).convert("RGB")
                folder = DATA_DIR / "assistant_uploads"
                folder.mkdir(exist_ok=True)
                chosen_path = folder / f"upload_{time.time_ns()}.png"
                uploaded_image.save(chosen_path)
            else:
                chosen_path = Path(image_choice.value)
            im = open_rgb(chosen_path); im.thumbnail((650, 420)); display(im)
            answer = ask_llava(chosen_path, question_box.value)
            print(answer)
            assistant_log.append({"image": str(chosen_path), "question": question_box.value,
                                  "answer": answer, "time_utc": pd.Timestamp.now(tz="UTC").isoformat()})
            with (RESULTS / "assistant_history.jsonl").open("a") as stream:
                stream.write(json.dumps(assistant_log[-1], ensure_ascii=False) + "\n")
        except Exception as exc:
            print(type(exc).__name__, ":", exc)
        finally:
            answer_button.disabled = llava is None

answer_button.on_click(run_assistant)
display(widgets.VBox([image_choice, image_upload, question_box, answer_button, assistant_output]))
if llava is None:
    print("Charger LLaVA pour activer l'assistant.")


### Bonus — CLIP retrieval + LLaVA

La requête récupère les images, puis LLaVA décrit chaque image séparément. Il ne
reçoit pas une collection multi-images non prévue par ce checkpoint. La description
peut confirmer ou mettre en doute le lien avec la requête ; elle ne prouve pas la pertinence.
Les embeddings existants évitent de réencoder toute la base après chargement de LLaVA.


In [ ]:
def retrieve_and_explain(query, k=3):
    table = retrieve_images(query, k=k)
    explanations = []
    for path in table["image"]:
        explanations.append(ask_llava(path, f"Describe the visible content, then explain whether it matches "
                                      f"this search query: {query}. State mismatches or uncertainty."))
    table["llava_explanation"] = explanations
    return table

RUN_BONUS = False  # Activer pour exécuter trois générations supplémentaires.
if RUN_BONUS and llava is not None:
    bonus_df = retrieve_and_explain("a photo of a cat", k=3)
    bonus_df.to_csv(RESULTS / "retrieval_llava_bonus.csv", index=False)
    display(bonus_df)


# Part III — Réponses aux 36 questions

Les réponses numériques ci-dessous renvoient aux sorties expérimentales. Elles
doivent être complétées après exécution, avec vos images et votre version de modèle.

| Question | Réponse / emplacement du résultat |
|---|---|
| **Q1** | CLIP (Contrastive Language–Image Pretraining) apprend à rapprocher les représentations d'images et de textes correspondants, et à éloigner les paires non correspondantes. |
| **Q2** | Un encodeur d'images et un encodeur de textes, avec projections vers un espace de représentation commun. |
| **Q3** | `CLIPProcessor` regroupe le prétraitement des images et la tokenisation des textes pour construire les tenseurs d'entrée. |
| **Q4** | `CLIPModel` calcule les représentations des deux modalités et leurs scores de compatibilité. |
| **Q5** | Lire `image.size` dans la section 2 : `(largeur, hauteur)`. Pour CIFAR-10, 32 × 32 pixels. |
| **Q6** | Une image PIL contient les pixels et des métadonnées. Après conversion RGB, chaque pixel possède trois composantes rouge, verte, bleue ; pour ces images usuelles, elles sont codées sur 8 bits chacune. L'objet PIL n'est pas encore un tenseur PyTorch. |
| **Q7** | Le texte en tête de `similarity_df`, affiché par la section 3. |
| **Q8** | Comparer ce texte à l'image réelle. Ne pas inférer la justesse à partir du seul score. |
| **Q9** | L'entraînement contrastif aligne les images et les textes dans le même espace, où un produit scalaire normalisé mesure leur proximité. |
| **Q10** | Le softmax transforme les logits en valeurs positives dont la somme vaut 1 : `p_i = exp(z_i)/sum_j exp(z_j)`. Ces valeurs dépendent de tous les candidats et ne sont pas automatiquement calibrées. |
| **Q11** | Reconnaître des classes à l'aide de leurs descriptions sans entraîner un nouveau classifieur sur des exemples annotés de ces classes pour ce TP. Cela ne signifie pas que les concepts étaient absents du préentraînement. |
| **Q12** | Le modèle préentraîné possède déjà un espace image–texte partagé ; on compare l'image aux descriptions de classes sans ajuster les poids. |
| **Q13** | CLIP choisit tout de même un candidat, éventuellement avec un score élevé. Sans mécanisme de rejet, il ne peut pas renvoyer une classe absente de la liste. |
| **Q14** | `clip_accuracy = nombre de prédictions correctes / nombre d'images`, calculée section 4 et sauvegardée dans `clip_classification.csv`. |
| **Q15** | Les lignes où `correct=False`, affichées section 4. Si aucune erreur n'est observée, le préciser pour cet échantillon uniquement. |
| **Q16** | Exemples : basse résolution, objet masqué ou ambigu, décalage avec les données de préentraînement, classes proches ou prompts peu adaptés. Examiner les erreurs pour déterminer les explications plausibles, sans les affirmer sans preuve. |
| **Q17** | Les scores ne sont généralement pas identiques ; vérifier les logits réels section 5. |
| **Q18** | Lire la ligne au logit maximal dans `prompt_comparison.csv`. Le meilleur prompt peut changer avec l'image. |
| **Q19** | La formulation change l'embedding de texte et donc le classement. Une amélioration systématique doit être évaluée sur un ensemble indépendant. |
| **Q20** | Le rang 1 de `retrieval_football_top5.csv`, affiché section 6. |
| **Q21** | Jugement humain de pertinence sémantique. Dans la base CIFAR seule, la requête football est hors du contenu prévu : un voisin le plus proche peut rester non pertinent. |
| **Q22** | L'encodeur préentraîné représente chaque image sans annotation de cette base ; la requête est comparée directement à ces représentations. Les annotations restent utiles pour évaluer la recherche. |
| **Q23** | Rechercher des photographies dans une photothèque à l'aide d'une description textuelle plutôt que de mots-clés saisis pour chaque image. |
| **Q24** | L'encodeur visuel extrait des représentations des régions/patches de l'image. |
| **Q25** | Le projecteur rend les représentations visuelles compatibles avec les dimensions et l'espace des embeddings du modèle de langage. Dans LLaVA 1.5, c'est un MLP. |
| **Q26** | Le modèle de langage conditionne sa génération sur les représentations visuelles et l'instruction pour produire une réponse textuelle. |
| **Q27** | Lire `vqa_accuracy_25` après les 25 notations humaines. Sans toutes les notes, la valeur complète reste absente. |
| **Q28** | Le comptage, les petits objets, le texte fin, les relations spatiales et les questions présupposant des objets absents peuvent être difficiles. Vérifier quelles catégories posent réellement problème dans vos réponses. |
| **Q29** | Le vélo doit être absent selon l'inspection humaine ; confirmer cette condition avant le test. |
| **Q30** | Examiner les réponses enregistrées : attribuer une couleur ou une position à un vélo absent constitue une hallucination. Si le modèle signale l'absence, il n'a pas halluciné sur ce point. |
| **Q31** | Une affirmation générée sur le contenu visuel qui n'est pas étayée par l'image, par exemple un objet absent présenté comme visible. |
| **Q32** | Des indices visuels incomplets, les associations apprises, la tendance à poursuivre une prémisse de la question et l'objectif de génération peuvent produire une réponse plausible mais fausse. |
| **Q33** | CLIP est généralement le choix direct pour une classification zero-shot sur une liste connue et pour calculer des scores comparables. Une supériorité de précision sur LLaVA doit être mesurée, pas supposée. |
| **Q34** | LLaVA, car il génère une description libre ; CLIP ne génère pas de phrases. |
| **Q35** | LLaVA pour des questions visuelles ouvertes. CLIP peut classer des réponses candidates, mais ne fournit pas directement une VQA libre. |
| **Q36** | CLIP apprend un espace de similarité image–texte avec deux encodeurs ; LLaVA associe un encodeur visuel, un projecteur et un modèle de langage génératif conditionné par l'image et le texte. |




## 14. Rapport et export des livrables

Relancer après les interactions et annotations. L'archive contient les tableaux,
embeddings, figures, paramètres, journal de l'assistant et les images effectivement
utilisées (dans `used_images/`, avec une table de correspondance), sans les poids des
modèles ni le téléchargement CIFAR complet. Les références VQA et les notes humaines
doivent être renseignées avant la remise finale. **Le ZIP n'inclut pas les sorties de
ce notebook :** télécharger également le `.ipynb` depuis Colab après exécution.


In [ ]:
save_manifest()
clip_table = pd.read_csv(RESULTS / "clip_classification.csv")
vqa_metric = None
vqa_done = 0
if VQA_FILE.exists():
    current_vqa = pd.read_csv(VQA_FILE)
    scores = pd.to_numeric(current_vqa["human_correct"], errors="raise")
    if not scores.dropna().isin([0, 1]).all():
        raise ValueError("Notes VQA invalides.")
    vqa_done = int(scores.notna().sum())
    if len(current_vqa) == 25 and vqa_done == 25:
        vqa_metric = float(scores.mean())

hallucination_metric = None
hallucination_attempts = 0
if HALLUCINATION_FILE.exists():
    current_hall = pd.read_csv(HALLUCINATION_FILE)
    hall_scores = pd.to_numeric(current_hall["human_hallucination"], errors="raise")
    if not hall_scores.dropna().isin([0, 1]).all():
        raise ValueError("Notes hallucination invalides.")
    hallucination_attempts = int(current_hall["attempt_id"].nunique())
    if hall_scores.notna().any():
        hallucination_metric = bool((hall_scores.dropna() == 1).any())
    if (hall_scores == 1).any():
        hall_text = "Au moins une hallucination a été identifiée par l'évaluation humaine."
    elif hall_scores.notna().all():
        hall_text = "Aucune hallucination observée dans les réponses de ce test, toutes évaluées."
    else:
        hall_text = "Test effectué ; annotation humaine d'hallucination encore incomplète."
else:
    hall_text = "Test d'hallucination non exécuté."

summary = {
    "clip_n": len(clip_table), "clip_accuracy": float(clip_table["correct"].mean()),
    "vqa_annotated": vqa_done, "vqa_accuracy_25": vqa_metric,
    "hallucination_observed_among_annotated": hallucination_metric,
    "hallucination_attempts": hallucination_attempts,
    "llava_status": manifest["llava_status"]
}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
vqa_text = f"{vqa_metric:.2%}" if vqa_metric is not None else f"non disponible ({vqa_done}/25 notes humaines)"
report = f"""# Rapport d'exécution — CLIP & LLaVA

## Configuration
Seed : {SEED}. CLIP : {CLIP_ID}. LLaVA : {LLAVA_ID}.
État LLaVA : {manifest['llava_status']}. Quantification demandée : {USE_4BIT}.
Voir run_manifest.json pour le matériel, les versions et les commits chargés.

## Résultats mesurés
- Classification CLIP : {len(clip_table)} images, accuracy {summary['clip_accuracy']:.2%}.
- VQA : {vqa_text}.
- Hallucination : {hall_text} Nombre de tentatives sauvegardées : {hallucination_attempts}.
- Recherche : inspecter retrieval_football_top5.csv et les figures ; la pertinence ne découle pas du score.

## Conclusion
CLIP permet de classer des descriptions candidates et d'indexer des images par similarité.
LLaVA permet de répondre librement à des questions visuelles lorsque son exécution aboutit.
Les observations qualitatives doivent être complétées avec les erreurs et exemples réellement vus.
Ces expériences pédagogiques ne suffisent pas à établir une supériorité générale d'un modèle.

## À compléter avant remise
Préciser les références humaines VQA, commenter les erreurs CLIP, juger les résultats de recherche,
et décrire un cas d'hallucination réel s'il a été observé. Joindre le notebook avec ses sorties.
"""
(RESULTS / "lab_report_fr.md").write_text(report, encoding="utf-8")
display(Markdown(report))

# Paquet reproductible : résultats + images effectivement utilisées.
package_dir = ROOT / "deliverable"
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir()
shutil.copytree(RESULTS, package_dir / "results")
images_dir = package_dir / "used_images"
images_dir.mkdir()
used_paths = set(retrieval_paths + [IMAGE_PATH] + [str(p) for p in vqa_paths])
for table_file in [VQA_FILE, HALLUCINATION_FILE]:
    if table_file.exists():
        used_paths.update(pd.read_csv(table_file)["image"].dropna().astype(str))
history_file = RESULTS / "assistant_history.jsonl"
if history_file.exists():
    for line in history_file.read_text().splitlines():
        used_paths.add(json.loads(line)["image"])
path_map = []
for i, original in enumerate(sorted(used_paths)):
    path = Path(original)
    if path.is_file():
        target = images_dir / f"{i:05d}{path.suffix.lower()}"
        shutil.copy2(path, target)
        path_map.append({"original_path": str(path), "archive_path": f"used_images/{target.name}",
                         "sha256": image_hash(path)})
pd.DataFrame(path_map).to_csv(package_dir / "image_path_mapping.csv", index=False)
archive = shutil.make_archive(str(ROOT / "CLIP_LLaVA_results"), "zip", root_dir=package_dir)
print("Archive prête :", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))
